In [1]:
import os
import sys
from pathlib import Path

import torch

REPO_ROOT = "/vol/biomedic3/tx1215/REPA-E"
FAILURE_DIR = "/vol/biomedic3/tx1215/REPA-E/checkpoints/embed_latent_dit_sitb2_256x192_debug_2/failures/step_0008468"
DEVICE = torch.device("cuda:0")

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from models.sit import SiT_models

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Failure dir:", FAILURE_DIR)

/vol/biomedic3/tx1215/REPA-E/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.3.1+cu121
CUDA: 12.1
GPU: NVIDIA GeForce RTX 3090
Failure dir: /vol/biomedic3/tx1215/REPA-E/checkpoints/embed_latent_dit_sitb2_256x192_debug_2/failures/step_0008468


In [2]:
def load_pt(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

ckpt = load_pt(Path(FAILURE_DIR) / "model_before_failure.pt")
rank0 = load_pt(Path(FAILURE_DIR) / "rank_0.pt")
rank1 = load_pt(Path(FAILURE_DIR) / "rank_1.pt")

args = ckpt["args"]

print("checkpoint keys:")
print(ckpt.keys())

print("\nrank snapshot keys:")
print(rank0.keys())

print("\nfailing step:", ckpt["failing_step"])
print("checkpoint steps:", ckpt["steps"])

print("\noriginal numerical settings:")
print("mixed_precision =", rank0["mixed_precision"])
print("compile         =", rank0["compile"])
print("fused_attn      =", rank0["fused_attn"])
print("qk_norm         =", rank0["qk_norm"])

print("\noriginal losses:")
print("rank0:", rank0["losses"])
print("rank1:", rank1["losses"])

print("\noriginal bad gradients:")
print("rank0:", rank0["bad_grad_stats"])
print("rank1:", rank1["bad_grad_stats"])

checkpoint keys:
dict_keys(['model', 'ema', 'opt', 'scheduler', 'args', 'steps', 'failing_step', 'reason'])

rank snapshot keys:
dict_keys(['failing_step', 'rank', 'reason', 'mixed_precision', 'compile', 'fused_attn', 'qk_norm', 'z', 'parents', 'zs', 'raw_image', 'batch', 'forward_rng_state', 'losses', 'bad_grad_stats', 'bad_grad_tensors'])

failing step: 8468
checkpoint steps: 8467

original numerical settings:
mixed_precision = bf16
compile         = False
fused_attn      = True
qk_norm         = False

original losses:
rank0: {'sit_loss': 1.5786021947860718, 'denoising_loss': 1.5786021947860718, 'proj_loss': 0.0}
rank1: {'sit_loss': 1.6193236112594604, 'denoising_loss': 1.6193236112594604, 'proj_loss': 0.0}

original bad gradients:
rank0: [{'name': 'blocks.8.mlp.fc2.weight', 'shape': (768, 2048), 'dtype': 'torch.float32', 'finite_absmax': 3.5859375, 'num_nan': 766, 'num_posinf': 0, 'num_neginf': 2, 'numel': 1572864}]
rank1: [{'name': 'blocks.8.mlp.fc2.weight', 'shape': (768, 2048), 

In [3]:
def describe(x, name="tensor"):
    if x is None:
        print(name, "= None")
        return
    if torch.is_tensor(x):
        finite = torch.isfinite(x)
        print(
            f"{name}: shape={tuple(x.shape)}, dtype={x.dtype}, "
            f"finite={finite.all().item()}, "
            f"min={x[finite].min().item() if finite.any() else float('nan'):.6g}, "
            f"max={x[finite].max().item() if finite.any() else float('nan'):.6g}, "
            f"mean={x[finite].float().mean().item() if finite.any() else float('nan'):.6g}, "
            f"nan={torch.isnan(x).sum().item()}, "
            f"inf={torch.isinf(x).sum().item()}"
        )
        return
    print(name, type(x))

describe(rank0["z"], "rank0 z")
describe(rank1["z"], "rank1 z")

print("\nrank0 parents:")
for k, v in rank0["parents"].items():
    describe(v, k)

print("\nrank1 parents:")
for k, v in rank1["parents"].items():
    describe(v, k)

rank0 z: shape=(64, 4, 32, 24), dtype=torch.float32, finite=True, min=-20.3562, max=22.691, mean=-1.35396, nan=0, inf=0
rank1 z: shape=(64, 4, 32, 24), dtype=torch.float32, finite=True, min=-21.0661, max=23.4543, mean=-1.3316, nan=0, inf=0

rank0 parents:
age: shape=(64, 1), dtype=torch.float32, finite=True, min=0.39853, max=0.89, mean=0.592721, nan=0, inf=0
view: shape=(64, 2), dtype=torch.float32, finite=True, min=0, max=1, mean=0.5, nan=0, inf=0
density: shape=(64, 4), dtype=torch.float32, finite=True, min=0, max=1, mean=0.25, nan=0, inf=0
scanner: shape=(64, 5), dtype=torch.float32, finite=True, min=0, max=1, mean=0.2, nan=0, inf=0

rank1 parents:
age: shape=(64, 1), dtype=torch.float32, finite=True, min=0.263387, max=0.860415, mean=0.590049, nan=0, inf=0
view: shape=(64, 2), dtype=torch.float32, finite=True, min=0, max=1, mean=0.5, nan=0, inf=0
density: shape=(64, 4), dtype=torch.float32, finite=True, min=0, max=1, mean=0.25, nan=0, inf=0
scanner: shape=(64, 5), dtype=torch.float3

In [4]:
def arg(name, default=None):
    return getattr(args, name, default)

def move_to(x, device=DEVICE):
    if torch.is_tensor(x):
        return x.to(device)
    if isinstance(x, dict):
        return {k: move_to(v, device) for k, v in x.items()}
    if isinstance(x, list):
        return [move_to(v, device) for v in x]
    if isinstance(x, tuple):
        return tuple(move_to(v, device) for v in x)
    return x

def restore_rng(snapshot):
    state = snapshot["forward_rng_state"]

    if state.get("cpu") is not None:
        torch.set_rng_state(state["cpu"])

    if state.get("cuda") is not None:
        torch.cuda.set_rng_state(state["cuda"], device=DEVICE)

def collect_bad_grads(model):
    bad = []

    for name, p in model.named_parameters():
        if p.grad is None:
            continue

        g = p.grad.detach()

        if torch.isfinite(g).all():
            continue

        finite = g[torch.isfinite(g)]

        bad.append({
            "name": name,
            "shape": tuple(g.shape),
            "dtype": str(g.dtype),
            "finite_absmax": finite.abs().max().item() if finite.numel() else float("nan"),
            "nan": torch.isnan(g).sum().item(),
            "+inf": torch.isposinf(g).sum().item(),
            "-inf": torch.isneginf(g).sum().item(),
            "numel": g.numel(),
        })

    return bad

In [5]:
def build_model(snapshot, fused_attn=None, qk_norm=None):
    if args.vae == "f8d4":
        latent_size = (args.height // 8, args.width // 8)
        in_channels = 4
    elif args.vae == "f16d32":
        latent_size = (args.height // 16, args.width // 16)
        in_channels = 32
    else:
        raise ValueError(args.vae)

    if args.proj_coeff > 0:
        raise RuntimeError(
            "This replay notebook currently assumes proj_coeff=0, "
            "which is the setting used for this failed run."
        )

    if fused_attn is None:
        fused_attn = snapshot["fused_attn"]

    if qk_norm is None:
        qk_norm = snapshot["qk_norm"]

    model = SiT_models[args.model](
        input_size=latent_size,
        in_channels=in_channels,
        class_dropout_prob=args.cfg_prob,
        z_dims=[],
        encoder_depth=args.encoder_depth,
        bn_momentum=args.bn_momentum,
        fused_attn=fused_attn,
        qk_norm=qk_norm,
    )

    missing, unexpected = model.load_state_dict(ckpt["model"], strict=False)

    print("missing:", missing)
    print("unexpected:", unexpected)

    assert len(missing) == 0
    assert len(unexpected) == 0

    model = model.to(DEVICE)
    model.train()
    model.bn.eval()

    return model

In [6]:
def precision_context(precision):
    if precision == "bf16":
        return torch.autocast("cuda", dtype=torch.bfloat16)

    if precision == "fp16":
        return torch.autocast("cuda", dtype=torch.float16)

    if precision == "fp32":
        return torch.autocast("cuda", enabled=False)

    raise ValueError(precision)

In [7]:
def replay(
    snapshot,
    precision="bf16",
    compile_model=None,
    fused_attn=None,
    qk_norm=None,
    verbose=True,
):
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    torch.backends.cuda.matmul.allow_tf32 = bool(args.allow_tf32)
    torch.backends.cudnn.allow_tf32 = bool(args.allow_tf32)

    model = build_model(
        snapshot,
        fused_attn=fused_attn,
        qk_norm=qk_norm,
    )

    if compile_model is None:
        compile_model = snapshot["compile"]

    if compile_model:
        model = torch.compile(
            model,
            backend="inductor",
            mode="default",
        )

    z = move_to(snapshot["z"])
    parents = move_to(snapshot["parents"])
    zs = move_to(snapshot["zs"])

    model.zero_grad(set_to_none=True)

    # Critical: restore the RNG to immediately before the failed forward.
    restore_rng(snapshot)

    with precision_context(precision):
        out = model(
            x=z,
            parents=parents,
            zs=zs,
            loss_kwargs=dict(
                weighting=args.weighting,
                path_type=args.path_type,
                prediction=args.prediction,
                align_only=False,
            ),
        )

        denoising_loss = out["denoising_loss"].mean()
        proj_loss = out["proj_loss"].mean()

        if args.proj_coeff > 0:
            loss = denoising_loss + args.proj_coeff * proj_loss
        else:
            loss = denoising_loss

    loss.backward()

    base_model = model._orig_mod if hasattr(model, "_orig_mod") else model
    bad = collect_bad_grads(base_model)

    result = {
        "rank": snapshot["rank"],
        "precision": precision,
        "compile": compile_model,
        "fused_attn": snapshot["fused_attn"] if fused_attn is None else fused_attn,
        "qk_norm": snapshot["qk_norm"] if qk_norm is None else qk_norm,
        "loss": loss.detach().float().item(),
        "saved_loss": snapshot["losses"]["sit_loss"],
        "bad_grads": bad,
    }

    if verbose:
        print("\nRESULT")
        for k, v in result.items():
            if k != "bad_grads":
                print(f"{k}: {v}")

        print("\nbad gradients:")
        if bad:
            for x in bad:
                print(x)
        else:
            print("NONE")

    return result, model

In [8]:
r0_bf16, _ = replay(
    rank0,
    precision="bf16",
)

Creating embedder for parent 'age' with input dim 1 and output dim 192
Creating embedder for parent 'view' with input dim 2 and output dim 192
Creating embedder for parent 'density' with input dim 4 and output dim 192
Creating embedder for parent 'scanner' with input dim 5 and output dim 192
missing: []
unexpected: []

RESULT
rank: 0
precision: bf16
compile: False
fused_attn: True
qk_norm: False
loss: 1.575270175933838
saved_loss: 1.5786021947860718

bad gradients:
NONE


In [9]:
r1_bf16, _ = replay(
    rank1,
    precision="bf16",
)

Creating embedder for parent 'age' with input dim 1 and output dim 192
Creating embedder for parent 'view' with input dim 2 and output dim 192
Creating embedder for parent 'density' with input dim 4 and output dim 192
Creating embedder for parent 'scanner' with input dim 5 and output dim 192
missing: []
unexpected: []

RESULT
rank: 1
precision: bf16
compile: False
fused_attn: True
qk_norm: False
loss: 1.6251857280731201
saved_loss: 1.6193236112594604

bad gradients:
NONE


In [10]:
r0_fp32, _ = replay(
    rank0,
    precision="fp32",
    compile_model=False,
)

r1_fp32, _ = replay(
    rank1,
    precision="fp32",
    compile_model=False,
)

Creating embedder for parent 'age' with input dim 1 and output dim 192
Creating embedder for parent 'view' with input dim 2 and output dim 192
Creating embedder for parent 'density' with input dim 4 and output dim 192
Creating embedder for parent 'scanner' with input dim 5 and output dim 192
missing: []
unexpected: []

RESULT
rank: 0
precision: fp32
compile: False
fused_attn: True
qk_norm: False
loss: 1.5682053565979004
saved_loss: 1.5786021947860718

bad gradients:
NONE
Creating embedder for parent 'age' with input dim 1 and output dim 192
Creating embedder for parent 'view' with input dim 2 and output dim 192
Creating embedder for parent 'density' with input dim 4 and output dim 192
Creating embedder for parent 'scanner' with input dim 5 and output dim 192
missing: []
unexpected: []

RESULT
rank: 1
precision: fp32
compile: False
fused_attn: True
qk_norm: False
loss: 1.6172592639923096
saved_loss: 1.6193236112594604

bad gradients:
NONE


In [11]:
def summary(result):
    return {
        "rank": result["rank"],
        "precision": result["precision"],
        "compile": result["compile"],
        "loss": result["loss"],
        "num_bad_params": len(result["bad_grads"]),
        "bad_names": [x["name"] for x in result["bad_grads"]],
    }

print(summary(r0_bf16))
print(summary(r1_bf16))
print(summary(r0_fp32))
print(summary(r1_fp32))

{'rank': 0, 'precision': 'bf16', 'compile': False, 'loss': 1.575270175933838, 'num_bad_params': 0, 'bad_names': []}
{'rank': 1, 'precision': 'bf16', 'compile': False, 'loss': 1.6251857280731201, 'num_bad_params': 0, 'bad_names': []}
{'rank': 0, 'precision': 'fp32', 'compile': False, 'loss': 1.5682053565979004, 'num_bad_params': 0, 'bad_names': []}
{'rank': 1, 'precision': 'fp32', 'compile': False, 'loss': 1.6172592639923096, 'num_bad_params': 0, 'bad_names': []}


In [12]:
print("Original compile setting:", rank0["compile"])

Original compile setting: False


In [13]:
def replay_with_fc2_hooks(
    snapshot,
    block_idx=8,
    precision="bf16",
):
    torch.cuda.empty_cache()

    model = build_model(snapshot)
    model.zero_grad(set_to_none=True)

    target = model.blocks[block_idx].mlp.fc2

    captured = {}

    def forward_hook(module, inputs, output):
        x = inputs[0]

        captured["fc2_input"] = x.detach().clone()
        captured["fc2_output"] = output.detach().clone()

        def output_grad_hook(grad):
            captured["fc2_grad_output"] = grad.detach().clone()

        output.register_hook(output_grad_hook)

    handle = target.register_forward_hook(forward_hook)

    z = move_to(snapshot["z"])
    parents = move_to(snapshot["parents"])
    zs = move_to(snapshot["zs"])

    restore_rng(snapshot)

    with precision_context(precision):
        out = model(
            x=z,
            parents=parents,
            zs=zs,
            loss_kwargs=dict(
                weighting=args.weighting,
                path_type=args.path_type,
                prediction=args.prediction,
                align_only=False,
            ),
        )

        loss = out["denoising_loss"].mean()

    loss.backward()

    handle.remove()

    captured["fc2_weight_grad"] = target.weight.grad.detach().clone()

    if target.bias is not None and target.bias.grad is not None:
        captured["fc2_bias_grad"] = target.bias.grad.detach().clone()

    return captured, loss.detach().float().item()

In [14]:
captured, loss = replay_with_fc2_hooks(
    rank0,
    block_idx=8,
    precision="bf16",
)

print("loss:", loss)

for name, x in captured.items():
    describe(x, name)

Creating embedder for parent 'age' with input dim 1 and output dim 192
Creating embedder for parent 'view' with input dim 2 and output dim 192
Creating embedder for parent 'density' with input dim 4 and output dim 192
Creating embedder for parent 'scanner' with input dim 5 and output dim 192
missing: []
unexpected: []
loss: 1.575270175933838
fc2_input: shape=(64, 192, 2048), dtype=torch.bfloat16, finite=True, min=-11008, max=64768, mean=60.7533, nan=0, inf=0
fc2_output: shape=(64, 192, 768), dtype=torch.bfloat16, finite=True, min=-7840, max=14528, mean=63.2758, nan=0, inf=0
fc2_grad_output: shape=(64, 192, 768), dtype=torch.bfloat16, finite=True, min=-4.30271e-07, max=6.85453e-07, mean=-3.79183e-11, nan=0, inf=0
fc2_weight_grad: shape=(768, 2048), dtype=torch.float32, finite=True, min=-15.6875, max=4.65625, mean=-2.84683e-05, nan=0, inf=0
fc2_bias_grad: shape=(768,), dtype=torch.float32, finite=True, min=-0.00028038, max=8.15392e-05, mean=-4.66785e-07, nan=0, inf=0


In [15]:
captured, loss = replay_with_fc2_hooks(
    rank1,
    block_idx=8,
    precision="bf16",
)

for name, x in captured.items():
    describe(x, name)

Creating embedder for parent 'age' with input dim 1 and output dim 192
Creating embedder for parent 'view' with input dim 2 and output dim 192
Creating embedder for parent 'density' with input dim 4 and output dim 192
Creating embedder for parent 'scanner' with input dim 5 and output dim 192
missing: []
unexpected: []
fc2_input: shape=(64, 192, 2048), dtype=torch.bfloat16, finite=True, min=-11136, max=65536, mean=61.9953, nan=0, inf=0
fc2_output: shape=(64, 192, 768), dtype=torch.bfloat16, finite=True, min=-7904, max=14656, mean=64.5379, nan=0, inf=0
fc2_grad_output: shape=(64, 192, 768), dtype=torch.bfloat16, finite=True, min=-4.32134e-07, max=6.07222e-07, mean=-1.33015e-11, nan=0, inf=0
fc2_weight_grad: shape=(768, 2048), dtype=torch.float32, finite=True, min=-9.3125, max=4.75, mean=-1.01558e-05, nan=0, inf=0
fc2_bias_grad: shape=(768,), dtype=torch.float32, finite=True, min=-0.000167847, max=8.39233e-05, mean=-1.63455e-07, nan=0, inf=0


In [16]:
x = captured["fc2_input"]
dy = captured["fc2_grad_output"]

print("x:", x.shape, x.dtype)
print("dy:", dy.shape, dy.dtype)

x2 = x.reshape(-1, x.shape[-1])
dy2 = dy.reshape(-1, dy.shape[-1])

print("flattened x:", x2.shape)
print("flattened dy:", dy2.shape)

x: torch.Size([64, 192, 2048]) torch.bfloat16
dy: torch.Size([64, 192, 768]) torch.bfloat16
flattened x: torch.Size([12288, 2048])
flattened dy: torch.Size([12288, 768])


In [17]:
dw_bf16 = dy2.transpose(0, 1) @ x2

describe(dw_bf16, "manual BF16 dW")

manual BF16 dW: shape=(768, 2048), dtype=torch.bfloat16, finite=True, min=-9.3125, max=4.75, mean=-1.01558e-05, nan=0, inf=0


In [18]:
dw_fp32 = dy2.float().transpose(0, 1) @ x2.float()

describe(dw_fp32, "manual FP32 dW")

manual FP32 dW: shape=(768, 2048), dtype=torch.float32, finite=True, min=-9.29829, max=4.76431, mean=-1.01455e-05, nan=0, inf=0


In [19]:
autograd_dw = captured["fc2_weight_grad"]

describe(autograd_dw, "autograd dW")

finite = torch.isfinite(autograd_dw)

if finite.any():
    diff = (
        autograd_dw[finite].float()
        - dw_fp32[finite].float()
    ).abs()

    print(
        "FP32 manual vs finite autograd entries:",
        "mean abs diff =", diff.mean().item(),
        "max abs diff =", diff.max().item(),
    )

autograd dW: shape=(768, 2048), dtype=torch.float32, finite=True, min=-9.3125, max=4.75, mean=-1.01558e-05, nan=0, inf=0
FP32 manual vs finite autograd entries: mean abs diff = 3.0273283755377633e-06 max abs diff = 0.014307498931884766


In [20]:
r0_no_fused, _ = replay(
    rank0,
    precision="bf16",
    compile_model=False,
    fused_attn=False,
)

print(summary(r0_no_fused))

Creating embedder for parent 'age' with input dim 1 and output dim 192
Creating embedder for parent 'view' with input dim 2 and output dim 192
Creating embedder for parent 'density' with input dim 4 and output dim 192
Creating embedder for parent 'scanner' with input dim 5 and output dim 192
missing: []
unexpected: []

RESULT
rank: 0
precision: bf16
compile: False
fused_attn: False
qk_norm: False
loss: 1.5741772651672363
saved_loss: 1.5786021947860718

bad gradients:
NONE
{'rank': 0, 'precision': 'bf16', 'compile': False, 'loss': 1.5741772651672363, 'num_bad_params': 0, 'bad_names': []}


In [21]:
saved_bad = rank0["bad_grad_tensors"]["blocks.8.mlp.fc2.weight"]

describe(saved_bad, "saved DDP bad gradient")

nan_positions = torch.isnan(saved_bad).nonzero()

print("number of NaN positions:", len(nan_positions))
print("first 30 NaN positions:")
print(nan_positions[:30])

saved DDP bad gradient: shape=(768, 2048), dtype=torch.float32, finite=False, min=-1.55469, max=3.58594, mean=6.57968e-06, nan=766, inf=2
number of NaN positions: 766
first 30 NaN positions:
tensor([[   0, 1246],
        [   1, 1246],
        [   2, 1246],
        [   3, 1246],
        [   4, 1246],
        [   5, 1246],
        [   6, 1246],
        [   7, 1246],
        [   8, 1246],
        [   9, 1246],
        [  10, 1246],
        [  11, 1246],
        [  12, 1246],
        [  13, 1246],
        [  14, 1246],
        [  15, 1246],
        [  16, 1246],
        [  17, 1246],
        [  18, 1246],
        [  19, 1246],
        [  20, 1246],
        [  21, 1246],
        [  22, 1246],
        [  23, 1246],
        [  24, 1246],
        [  25, 1246],
        [  26, 1246],
        [  27, 1246],
        [  28, 1246],
        [  29, 1246]])


In [22]:
row_nan_counts = torch.isnan(saved_bad).sum(dim=1)
col_nan_counts = torch.isnan(saved_bad).sum(dim=0)

print("rows containing NaNs:", (row_nan_counts > 0).sum().item())
print("columns containing NaNs:", (col_nan_counts > 0).sum().item())

print("\nlargest row NaN counts:")
print(torch.topk(row_nan_counts, k=20))

print("\nlargest column NaN counts:")
print(torch.topk(col_nan_counts, k=20))

rows containing NaNs: 766
columns containing NaNs: 1

largest row NaN counts:
torch.return_types.topk(
values=tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
indices=tensor([512, 518, 522, 523, 524, 525, 526, 521, 514, 513, 515, 511, 510, 509,
        508, 507, 506, 505, 504, 516]))

largest column NaN counts:
torch.return_types.topk(
values=tensor([766,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0]),
indices=tensor([1246,    4,   18,   10,   16,    8,   19,    1,    5,   11,   12,   15,
           7,   17,    3,    9,   13,    0,    2,    6]))


In [23]:
print("\n=== REPLAY SUMMARY ===")
print("rank0 BF16:", summary(r0_bf16))
print("rank1 BF16:", summary(r1_bf16))
print("rank0 FP32:", summary(r0_fp32))
print("rank1 FP32:", summary(r1_fp32))

print("\n=== FC2 ===")
describe(captured["fc2_input"], "fc2_input")
describe(captured["fc2_grad_output"], "fc2_grad_output")
describe(captured["fc2_weight_grad"], "fc2_weight_grad")

print("\n=== MANUAL MATMUL ===")
describe(dw_bf16, "manual BF16 dW")
describe(dw_fp32, "manual FP32 dW")

print("\n=== NO FUSED ATTENTION ===")
print(summary(r0_no_fused))

print("\n=== SAVED FAILURE ===")
describe(saved_bad, "saved DDP bad gradient")
print("rows with NaN:", (row_nan_counts > 0).sum().item())
print("cols with NaN:", (col_nan_counts > 0).sum().item())
print("total NaNs:", torch.isnan(saved_bad).sum().item())


=== REPLAY SUMMARY ===
rank0 BF16: {'rank': 0, 'precision': 'bf16', 'compile': False, 'loss': 1.575270175933838, 'num_bad_params': 0, 'bad_names': []}
rank1 BF16: {'rank': 1, 'precision': 'bf16', 'compile': False, 'loss': 1.6251857280731201, 'num_bad_params': 0, 'bad_names': []}
rank0 FP32: {'rank': 0, 'precision': 'fp32', 'compile': False, 'loss': 1.5682053565979004, 'num_bad_params': 0, 'bad_names': []}
rank1 FP32: {'rank': 1, 'precision': 'fp32', 'compile': False, 'loss': 1.6172592639923096, 'num_bad_params': 0, 'bad_names': []}

=== FC2 ===
fc2_input: shape=(64, 192, 2048), dtype=torch.bfloat16, finite=True, min=-11136, max=65536, mean=61.9953, nan=0, inf=0
fc2_grad_output: shape=(64, 192, 768), dtype=torch.bfloat16, finite=True, min=-4.32134e-07, max=6.07222e-07, mean=-1.33015e-11, nan=0, inf=0
fc2_weight_grad: shape=(768, 2048), dtype=torch.float32, finite=True, min=-9.3125, max=4.75, mean=-1.01558e-05, nan=0, inf=0

=== MANUAL MATMUL ===
manual BF16 dW: shape=(768, 2048), dtype

In [24]:
print("rank0")
print("saved :", r0_bf16["saved_loss"])
print("replay:", r0_bf16["loss"])
print("diff  :", abs(r0_bf16["saved_loss"] - r0_bf16["loss"]))

print("\nrank1")
print("saved :", r1_bf16["saved_loss"])
print("replay:", r1_bf16["loss"])
print("diff  :", abs(r1_bf16["saved_loss"] - r1_bf16["loss"]))

rank0
saved : 1.5786021947860718
replay: 1.575270175933838
diff  : 0.0033320188522338867

rank1
saved : 1.6193236112594604
replay: 1.6251857280731201
diff  : 0.005862116813659668
